# A3C / A2C — Asynchronous Methods for Deep Reinforcement Learning (2016)

_Papers · Reinforcement Learning_

**Train a policy and a value critic together with the advantage A = R − V, plus an entropy bonus — A3C runs many actors in parallel; A2C is the synchronous version of the same update.**

---

This notebook builds advantage actor-critic from scratch, one piece at a time: the network, the n-step return, the A2C update, and a CartPole training loop. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

### Step 1 — Sanity-check the worked example by hand

Before building anything, we reproduce the lesson's small numerical example so the rest of the code has a reference point. We form a 2-step bootstrapped return `R_t`, subtract the critic's baseline `V(s_t)` to get the advantage `A`, and take the log-probability of the action the policy chose. The per-sample policy loss term `-log(pi) * A` is what advantage actor-critic minimizes; a positive advantage makes the loss push the action's probability up.

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import math
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)

# Sanity-check the lesson's worked example.
gamma = 0.99
R_t = 1 + gamma * 1 + gamma**2 * 19.5      # 2-step bootstrapped return
A_t = R_t - 18.0                           # advantage = return - baseline V(s_t)
logp = math.log(0.6)                       # log-prob of the taken action (pi = 0.6)
loss_term = -logp * A_t                    # per-sample policy loss term
print("worked example:  R_t =", round(R_t, 4), " A =", round(A_t, 4),
      " log(0.6) =", round(logp, 4), " -log(pi)*A =", round(loss_term, 4))
# worked example:  R_t = 21.1019  A = 3.1019  log(0.6) = -0.5108  -log(pi)*A = 1.5846

### Step 2 — Build the actor-critic network and the n-step return

The paper shares most parameters between the actor and the critic, so the network has one shared body and two heads: a policy head that outputs action logits and a value head that estimates `V(s)`. The bootstrapped n-step return is computed backward — start from `V(s_last)` to bootstrap the tail, then sweep in reverse accumulating `R <- r + gamma*R`, resetting to 0 at episode boundaries. These two pieces are the actor-critic and the target signal the critic regresses toward.

In [ ]:
# Shared-body actor-critic network (Track B: nn.Linear primitives).
# Paper (S4): "we always share some of the parameters in practice."
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=128):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh())
        self.pi   = nn.Linear(hidden, n_act)   # policy head -> action logits (actor)
        self.v    = nn.Linear(hidden, 1)        # value head  -> V(s)        (critic)

    def forward(self, x):
        h = self.body(x)
        return Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)


# Bootstrapped n-step return, computed BACKWARD (Algorithm S3): R <- r + gamma R.
def compute_returns(rewards, dones, last_v, gamma=0.99):
    R = last_v                                  # bootstrap the tail with V(s_last)...
    out = [0.0] * len(rewards)
    for t in reversed(range(len(rewards))):
        if dones[t]:                            # ...but reset to 0 across episode ends
            R = 0.0
        R = rewards[t] + gamma * R
        out[t] = R
    return torch.tensor(out, dtype=torch.float32)

### Step 3 — Define one advantage actor-critic update

A single update combines three terms from §4: the policy-gradient loss weighted by the advantage, the critic's squared-error value loss, and an entropy bonus that keeps the policy exploring. The advantage `A = R - V(s)` is detached when it weights the policy loss, so only the value loss trains the critic; an ablation flag lets us swap in the raw return to drop the baseline. We normalize the advantage, clip the gradient norm for stability, and step the optimizer.

In [ ]:
# One advantage actor-critic update: policy grad + value loss + entropy (S4).
def a2c_update(net, opt, obs, acts, returns, beta=0.01, use_baseline=True):
    dist, value = net(obs)
    logp = dist.log_prob(acts)
    if use_baseline:
        adv = returns - value                   # advantage A = R - V(s)   (S4)
    else:
        adv = returns                           # ABLATION: raw return, no baseline
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    policy_loss = -(logp * adv.detach()).mean() # detach: only the value loss trains V
    value_loss  = (returns - value).pow(2).mean()       # critic squared error (R - V)^2
    entropy     = dist.entropy().mean()                  # H(pi): entropy bonus
    loss = policy_loss + 0.5 * value_loss - beta * entropy
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), 0.5)
    opt.step()

### Step 4 — Train on CartPole and run the ablation

The training loop collects an on-policy rollout each update, bootstraps the tail value if the last step did not end an episode, computes returns, and applies one A2C update. We track the average return over the last 20 episodes and stop early once CartPole is solved. Running it twice — once with the advantage baseline and once with the raw return — shows the baselined agent climbing smoothly while the no-baseline ablation is noisier and slower.

In [ ]:
# Train until CartPole's return rises and it solves the task; PRINT the climb.
def train(use_baseline=True, updates=120, steps_per=512):
    torch.manual_seed(0)
    env = gym.make("CartPole-v1")
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    obs, _ = env.reset(seed=0)
    ep_ret, recent, history = 0.0, [], []
    for up in range(updates):
        O, Ac, R, D = [], [], [], []
        for _ in range(steps_per):              # collect an ON-POLICY rollout
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                dist, _ = net(ot)
                a = dist.sample()
            nobs, rew, term, trunc, _ = env.step(int(a))
            done = term or trunc
            O.append(ot); Ac.append(a); R.append(float(rew)); D.append(done)
            ep_ret += rew; obs = nobs
            if done:
                recent.append(ep_ret); ep_ret = 0.0
                obs, _ = env.reset()
        with torch.no_grad():
            last_v = 0.0 if D[-1] else float(net(torch.as_tensor(obs, dtype=torch.float32))[1])
        returns = compute_returns(R, D, last_v)
        a2c_update(net, opt, torch.stack(O), torch.stack(Ac), returns, use_baseline=use_baseline)
        avg = sum(recent[-20:]) / max(1, len(recent[-20:]))
        history.append(avg)
        print(f"  update {up:3d}  avg return (last 20 eps): {avg:6.1f}")
        recent = recent[-20:]
        if avg >= 475:
            print("  -> SOLVED CartPole."); break
    env.close()
    return history

print("Advantage actor-critic (A = R - V):")
adv_hist = train(use_baseline=True)
print("\nABLATION -- no baseline (raw return R, same everything else):")
raw_hist = train(use_baseline=False)
print("\nAdvantage avg-return trajectory:", [round(h, 1) for h in adv_hist])
print("No-baseline avg-return trajectory:", [round(h, 1) for h in raw_hist])
# The advantage agent climbs smoothly toward ~500 and solves CartPole; the no-baseline
# ablation is noisier and slower. (Exact numbers vary by hardware/seed; our small run,
# not the paper's reported Atari Table-1 number.)

## Visualize the data & results

_Does advantage actor-critic (A = R − V) make the CartPole return rise to the solved score, and does removing the baseline (raw return R, same everything else) make learning noisier and slower? We train both for the same updates and plot the average return per update._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train advantage actor-critic and the no-baseline ablation on CartPole-v1 for the same
# number of updates with identical net / returns / lr / entropy / seed, recording avg
# return per update.
#
#   adv_hist = train(use_baseline=True)    # green: A = R - V; climbs to ~500 (SOLVED)
#   raw_hist = train(use_baseline=False)   # red:   raw R, no baseline; noisier, slower
#
# The points plotted are the per-update average return (last 20 episodes).
# Advantage  -> smooth monotone climb past the 475 "solved" line.
# No-baseline -> rises but jitters: the raw return's high variance makes each policy-
# gradient step noisy, so learning is slower and less stable.
# (Numbers are from our own run; the paper reports a 496.8% Atari mean for A3C in
#  Table 1 vs 463.6% for Prioritized DQN, not these CartPole values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked advantage + policy-gradient term. With $\gamma = 0.99$, a 2-step rollout gives
            rewards $r_t = 1$, $r_{t+1} = 1$, ending at $s_{t+2}$ with critic values $V(s_t) = 18.0$ and
            $V(s_{t+2}) = 19.5$. The policy gave the taken action probability $\pi = 0.6$. Compute the
            bootstrapped return $R_t$, the advantage $A$, and the per-sample policy loss term
            $-\log\pi \cdot A$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bootstrapped return: $R_t = 1 + 0.99(1) + 0.99^2(19.5) = 21.1019$. — _Real rewards for $k=2$ steps plus the discounted critic bootstrap of the tail (Algorithm S3 / §4)._
- Advantage: $A = R_t - V(s_t) = 21.1019 - 18.0 = 3.1019$. — _The advantage is the return minus the baseline; positive means the action beat the critic's expectation._
- Log-prob: $\log\pi = \log 0.6 = -0.5108$. — _The policy gradient uses the log-probability of the action actually taken._
- Policy loss term: $-\log\pi \cdot A = -(-0.5108)(3.1019) = +1.5846$. — _We minimize the negative of the objective $\log\pi\cdot A$; descending it raises $\log\pi$ because $A\gt 0$._

**Answer:** $R_t = 21.1019$, $A = 3.1019$, and the per-sample policy loss term is $+1.5846$. Because the
                 advantage is positive, gradient descent on this loss raises the action's log-probability —
                 making the good action more likely next time. The notebook recomputes
                 $1 + 0.99 + 0.99^2\cdot 19.5 = 21.1019$, $A = 3.1019$, and $-\log(0.6)\cdot A = 1.5846$.

</details>

**Problem 2.** The ablation. Your advantage actor-critic solves CartPole (return climbs toward ~500).
            Replace the advantage $A = R - V$ in the policy loss with the raw return $R$ (no baseline) —
            keeping the network, returns, entropy, learning rate, and seed identical — and retrain. What happens
            to the return curve, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the policy weight: use -(logp * R.detach()).mean() instead of -(logp * adv.detach()).mean(); keep everything else fixed. — _An honest ablation changes exactly one thing — the baseline — so any difference is attributable to it._
- Retrain and watch the return: the no-baseline run is noisier and rises more slowly / less reliably than the advantage run. — _The raw return $R$ has high variance; the policy gradient weighted by it is noisy, so updates jitter. Subtracting $V(s)$ centers the signal and shrinks the variance._
- Conclude the baseline (the "advantage" in advantage actor-critic), not the network, is what makes learning fast and stable. — _Both runs share architecture, returns, and seed; only the baselined one is smooth, isolating the baseline as the cause._

**Answer:** The no-baseline (raw-return) agent learns more slowly and with a noisier, more jagged return
                 curve, while the advantage agent climbs smoothly toward ~500. Since the only difference is
                 subtracting $V(s)$, this isolates the baseline as the variance-reducer that makes
                 advantage actor-critic efficient — the "A" in A3C/A2C. The CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Suppose at some step the advantage came out negative, $A = -1.5$, for an action with
            probability $\pi = 0.7$ ($\log\pi = -0.3567$). What is the per-sample policy loss term
            $-\log\pi \cdot A$, and which way does the update move this action's probability?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Policy loss term: $-\log\pi \cdot A = -(-0.3567)(-1.5) = -0.5350$. — _Multiply the negated log-prob by the advantage; two negatives in $-\log\pi$ and one in $A$ give a negative product._
- Minimizing a negative loss term means the optimizer can lower it further by lowering $\log\pi$. — _Since $A\lt 0$, the gradient of $-\log\pi\cdot A$ pushes $\log\pi$ down — the opposite direction from the positive-advantage case._

**Answer:** The per-sample policy loss term is $-0.5350$, and because the advantage is negative the
                 update lowers this action's probability — the actor learns to take it less often. This
                 is the symmetry of the advantage: positive advantage pushes an action up, negative pushes it
                 down, with the magnitude set by how far the return beat or missed the critic's baseline.

</details>